In [2]:
%load_ext sql
%sql sqlite:///mydata.db

## SQL Logical Execution Order

We write a query starting with `SELECT`. But the database does not run it in that order — it always runs queries in this fixed order:

- **FROM** — pick the table(s) to read data from
- **JOIN** — combine tables together, if any joins are used
- **WHERE** — remove rows that don't match the condition
- **GROUP BY** — put the remaining rows into groups
- **HAVING** — remove whole groups that don't match the condition
- **SELECT** — pick the columns to show in the output
- **DISTINCT** — remove duplicate rows, if asked
- **ORDER BY** — sort the final rows
- **LIMIT** — cut down the number of rows shown (this is SQLite's version of MSSQL's `TOP`)

Why this order matters:

- Most confusing SQL errors for beginners happen simply because they don't know this order.
- You **cannot** use a column nickname (alias) from `SELECT` inside `WHERE`. This is because `WHERE` runs *before* `SELECT`, so that nickname doesn't exist yet at that point.
- You **cannot** use an aggregate function like `SUM()` or `COUNT()` inside `WHERE`. This is because `GROUP BY` (which creates the groups those functions work on) hasn't run yet when `WHERE` runs.

## SELECT Basics, and TOP Becoming LIMIT

What each SELECT keyword does:

- `SELECT *` — shows every column
- `SELECT DISTINCT city` — shows only unique values (no repeats)
- `SELECT COUNT(*)` — counts every row, even ones where some columns are NULL (empty)
- `SELECT COUNT(col)` — counts only the rows where that specific column is NOT NULL

Getting a limited number of rows — TOP becomes LIMIT:

- In MSSQL, you used `TOP N` to get only N rows. It went right after `SELECT`, before the column names.
- SQLite does **not** have `TOP` at all.
- Instead, SQLite uses `LIMIT N`.
- Key difference: `LIMIT` goes at the very **end** of the query, after `ORDER BY` — not near the start like `TOP` did.

In [7]:
%%sql
SELECT *
FROM orders
LIMIT 3

 * sqlite:///mydata.db
Done.


row_id,order_id,order_date,ship_date,ship_mode,customer_id,customer_name,segment,country,city,state,postal_code,region,product_id,category,sub_category,product_name,sales,quantity,discount,profit
1364.0,US-2021-155425,2021-11-10,2021-11-11,First Class,AB-10600,Ann Blume,Corporate,United States,Tucson,Arizona,85705.0,West,OFF-BI-10001036,Office Supplies,Binders,Cardinal EasyOpen D-Ring Binders,38.388,14.0,0.7,-25.592
1365.0,US-2021-155425,2021-11-10,2021-11-11,First Class,AB-10600,Ann Blume,Corporate,United States,Tucson,Arizona,85705.0,West,TEC-MA-10003183,Technology,Machines,DYMO CardScan Personal V9 Business Card Scanner,95.994,2.0,0.7,-63.996
1366.0,US-2021-155425,2021-11-10,2021-11-11,First Class,AB-10600,Ann Blume,Corporate,United States,Tucson,Arizona,85705.0,West,TEC-AC-10001314,Technology,Accessories,Case Logic 2.4GHz Wireless Keyboard,239.952,6.0,0.2,-35.9928


**LIMIT with OFFSET — getting a "page" of results**

This is useful when you want to skip some rows and grab the next batch (like page 3 of a list).

- `ORDER BY salary DESC` — first, sort employees from highest to lowest salary
- `OFFSET 20` — skip the first 20 rows (don't show them)
- `LIMIT 10` — after skipping, show only the next 10 rows
- So together, `LIMIT 10 OFFSET 20` means: *"skip 20, then give me 10"*

Comparison with MSSQL:

- MSSQL needed a longer way to write this: `OFFSET 20 ROWS FETCH NEXT 10 ROWS ONLY`
- SQLite does the same thing in one short, simple clause: `LIMIT 10 OFFSET 20`

In [8]:
%%sql
SELECT *
FROM orders
LIMIT 3 OFFSET 2

 * sqlite:///mydata.db
Done.


row_id,order_id,order_date,ship_date,ship_mode,customer_id,customer_name,segment,country,city,state,postal_code,region,product_id,category,sub_category,product_name,sales,quantity,discount,profit
1366.0,US-2021-155425,2021-11-10,2021-11-11,First Class,AB-10600,Ann Blume,Corporate,United States,Tucson,Arizona,85705.0,West,TEC-AC-10001314,Technology,Accessories,Case Logic 2.4GHz Wireless Keyboard,239.952,6.0,0.2,-35.9928
1367.0,US-2021-155425,2021-11-10,2021-11-11,First Class,AB-10600,Ann Blume,Corporate,United States,Tucson,Arizona,85705.0,West,TEC-PH-10002563,Technology,Phones,Adtran 1202752G1,201.584,2.0,0.2,15.1188
1368.0,US-2021-155425,2021-11-10,2021-11-11,First Class,AB-10600,Ann Blume,Corporate,United States,Tucson,Arizona,85705.0,West,FUR-CH-10003312,Furniture,Chairs,Hon 2090 �Pillow Soft� Series Mid Back Swivel/Tilt Chairs,899.136,4.0,0.2,-146.11


## WHERE Clause Operators

Basic comparison operators — no change needed:

- `=`, `<>`, `>`, `<`, `>=`, `<=` — these work the same in SQLite and MSSQL
- `BETWEEN ... AND` — same in both
- `IN`, `NOT IN` — same in both
- `IS NULL`, `IS NOT NULL` — same in both
- `EXISTS` — same in both
- So nothing here needs to be converted or changed.

**Rule 1 — always use single quotes for text values**

- Use `'South'` — with single quotes
- Do **not** use `"South"` — with double quotes
- Reason: in SQLite, double quotes are meant for naming things (like column names or table names), not for text values.
- Risky part: if you accidentally write `"South"` and there's no column or table named South, SQLite won't show an error — it will quietly just treat it as the text "South" anyway. This can hide mistakes instead of warning you.

In [10]:
%%sql
SELECT *
FROM orders
WHERE region = 'South' AND discount > 0
LIMIT 3

 * sqlite:///mydata.db
Done.


row_id,order_id,order_date,ship_date,ship_mode,customer_id,customer_name,segment,country,city,state,postal_code,region,product_id,category,sub_category,product_name,sales,quantity,discount,profit
8318.0,CA-2021-130904,2021-04-11,2021-04-16,Standard Class,HM-14980,Henry MacAllister,Consumer,United States,Burlington,North Carolina,27217.0,South,OFF-AR-10000422,Office Supplies,Art,Pencil and Crayon Sharpener,1.752,1.0,0.2,0.1533
8319.0,CA-2021-130904,2021-04-11,2021-04-16,Standard Class,HM-14980,Henry MacAllister,Consumer,United States,Burlington,North Carolina,27217.0,South,OFF-AR-10000127,Office Supplies,Art,Newell 321,20.992,8.0,0.2,2.3616
8325.0,CA-2021-144456,2021-09-08,2021-09-09,First Class,FC-14245,Frank Carlisle,Home Office,United States,Hialeah,Florida,33012.0,South,OFF-ST-10001321,Office Supplies,Storage,"Decoflex Hanging Personal Folder File, Blue",61.68,5.0,0.2,5.397
